# Eksperimen konsistensi label dengan pengulangan lirik

Buka dari root **SongMap**. Input, probabilitas model asal dan hasil yang sudah dijalankan disertakan. Anda dapat membaca ringkasan ini atau **Run All** untuk mengulang perbandingan tanpa ASR, audio, model inference, instalasi paket atau anotasi baru.

Aturan hanya menilai kandidat pertukaran **bridge ↔ chorus**, mempertahankan setiap boundary Apple dan jumlah section. Kandidat memerlukan cakupan frasa kuat, minimal dua frasa berbeda yang berulang pada waktu berbeda, serta section acuan yang **sejak awal** sudah diprediksi model sesuai lirik. Label kandidat tidak pernah menjadi acuan untuk koreksi lain.

Ini simulasi review lokal dengan referensi lirik yang Anda berikan, bukan keluaran model audio-only baru atau koreksi yang sudah dipasang di iPhone.


Hasil I Surrender yang sudah dijalankan:

- **04:32.52–04:52.77:** kandidat **chorus → bridge**; dua frasa berbeda memiliki pengulangan pada section acuan asli.
- **03:34–04:13:** masih ditahan, meskipun audit frasa menandainya sebagai bridge. Syarat dua pengulangan frasa kuat pada acuan asli belum terpenuhi.
- Section instrumental/campuran dan bagian terakhir tetap dipertahankan.
- **Seluruh waktu start/end dan jumlah section identik dengan input.**

Jadi percobaan ini belum menyelesaikan seluruh salah label. Angka similarity/coverage tidak diperlakukan sebagai confidence model baru. Hasil tersimpan di `experiments/structure/lyric_review_examples/i-surrender-label-experiment.json`.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess,sys,json

ROOT=Path.cwd().resolve()
FOLDER=ROOT/'experiments/structure/lyric_review_examples'
INPUT=FOLDER/'i-surrender.json'
MODEL=FOLDER/'i-surrender-model.json'
SCRIPT=ROOT/'scripts/experiment_lyric_labels.py'
assert SCRIPT.is_file() and INPUT.is_file() and MODEL.is_file(), 'Buka dari root SongMap setelah pull branch worship-hybrid-integration'
OUTPUT=ROOT/'outputs.local'/('lyric-label-experiment-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
log=Path(str(OUTPUT)+'.log');log.parent.mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u',str(SCRIPT),'--input',str(INPUT),'--model-evidence',str(MODEL),'--output-dir',str(OUTPUT)]
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:raise RuntimeError(f'Gagal ({status}); lihat {log}')


In [ ]:
from IPython.display import display,Markdown
result=json.loads((OUTPUT/'comparison.json').read_text())
def clock(s):return f'{int(s//60):02d}:{s%60:05.2f}'
lines=['| Waktu tetap | Label awal | Kandidat | Cakupan frasa kuat | Frasa berulang berbeda |',
       '|---|---|---|---|---|']
for row in result['decisions']:
    if row['status']!='candidate_label_change':continue
    lines.append(f"| {clock(row['start'])}–{clock(row['end'])} | {row['original_label']} | {row['candidate_label']} | {row['strong_match_coverage']:.1%} | {row['unique_repeated_phrases']} |")
display(Markdown('\n'.join(lines)))
print('Kandidat perubahan:',result['candidate_change_count'])
print('Boundary identik:',result['boundaries_exactly_preserved'])
print('Akurasi belum diukur pada ground truth independen.')
print('Hasil:',OUTPUT/'comparison.json')
print('CSV:',OUTPUT/'comparison.csv')


In [ ]:
# Tinjau keputusan yang ditahan dan bukti untuk kandidat yang lolos.
for row in result['decisions']:
    if row['matched_role'] is None or row['matched_role']==row['original_label']:continue
    print(f"\n{clock(row['start'])}–{clock(row['end'])}: {row['original_label']} → {row['candidate_label']}; {row['status']}")
    for support in row['support']:
        print(f"  Acuan asli section {support['anchor_section']}: {support['reference_phrase']}")
        print('  Waktu frasa acuan:',support['anchor_occurrence'],'| pengulangan:',support['target_occurrence'])


Probabilitas model asal diperiksa terhadap label, urutan kelas, sumber rekaman dan rentang yang sama. Probabilitasnya tidak diubah atau diklaim sebagai confidence kandidat. Data ini tidak menjalankan training, mengubah anotasi manual, mengaktifkan aturan di aplikasi atau mengukur speedup/akurasi iPhone.

Tahap evaluasi lanjutan membutuhkan contoh lagu lain dengan hasil ASR dan referensi yang sesuai rekamannya. Section campuran tetap menjadi eksperimen segmentasi terpisah; jangan mengubah seluruh blok instrumental menjadi chorus hanya karena satu frasa vokal di dalamnya.
